In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# Chargement
df = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv", sep=';', encoding='utf-8-sig')

# Cible
y = df["Montant"]
X = df.drop(columns=["Montant", "dataloadingdate", "JourSemaine"])  # adapte si besoin

# Encodage si besoin
X = pd.get_dummies(X, drop_first=True)

# Séparation



In [3]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"✅ RMSE : {rmse:.4f}")
    return rmse


In [4]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR()
}


In [6]:
print("🔹 ÉTAPE 1 : Modèles bruts")
rmse_scores = {}
for name, model in models.items():
    print(f"\n🔸 {name}")


🔹 ÉTAPE 1 : Modèles bruts

🔸 Linear Regression

🔸 Ridge Regression

🔸 Random Forest

🔸 Gradient Boosting

🔸 SVR


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n🔹 ÉTAPE 2 : Normalisation")
rmse_scaled = {}
for name, model in models.items():
    print(f"\n🔸 {name}")
    rmse_scaled[name] = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test)


In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

selector = SelectKBest(score_func=f_regression, k='all')
X_new = selector.fit_transform(X, y)
X_train_fs, X_test_fs, y_train_fs, y_test_fs = train_test_split(X_new, y, test_size=0.2, random_state=42)


print("\n🔹 ÉTAPE 3 : Feature Selection")
rmse_fs = {}
for name, model in models.items():
    print(f"\n🔸 {name}")
    rmse_fs[name] = evaluate_model(model, X_train_fs, y_train_fs, X_test_fs, y_test_fs)



In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20]
}

grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3, scoring='neg_root_mean_squared_error')
grid_search.fit(X_train, y_train)

print("🔹 ÉTAPE 4 : Hyperparameter Tuning - Random Forest")
print("Meilleurs paramètres :", grid_search.best_params_)

best_rf = grid_search.best_estimator_
rmse_tuned = evaluate_model(best_rf, X_train, y_train, X_test, y_test)


In [ ]:
print("\n=== RÉSUMÉ DES RMSE ===")
print("Brut :", rmse_scores)
print("Normalisé :", rmse_scaled)
print("Feature Selection :", rmse_fs)
print("Fine-tuning RF :", rmse_tuned)
